# 04 · Milestone 2: train the BERT → CLIP projection

SD 1.5's U-Net only understands CLIP ViT-L/14 hidden states. To condition it on **BERT**, this notebook trains a small Transformer adapter that maps BERT's `[77, 768]` token states onto CLIP's, using all ~25k COCO val caption pairs. Run notebook 01 first, because the manifest must exist on Drive.

How the evaluation is kept honest:
- The split is **by image**, so the 5 captions of one image never sit on both sides.
- The 2 COCO prompt images used for generation are **excluded**.
- Cosine is reported on **content tokens** as well as on all 77 positions. Padding positions look alike for every caption, so they inflate the all-token number.
- The adapter is compared against two baselines: predicting the mean CLIP embedding, and a closed-form linear map.

Runtime is about 5–10 min on a T4. Rerunning reuses the saved checkpoint unless `RETRAIN = True`.

In [ ]:
# --- Setup: mount Drive, clone or update the repo, install pinned deps. Safe to rerun. ---
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Pranav1011/ie7615-generative-project.git"
BRANCH = "main"
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

def _git(*args):
    """Run git. For a private repo, add a GITHUB_TOKEN Colab secret (key icon in the sidebar).

    The token goes in a one-off HTTP header, so it is never written to .git/config."""
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    auth = []
    if token:
        import base64
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        auth = ["-c", f"http.https://github.com/.extraheader=Authorization: Basic {basic}"]
    res = subprocess.run(["git", *auth, *args], capture_output=True, text=True)
    out = res.stdout + res.stderr
    if token:
        out = out.replace(token, "***").replace(basic, "***")
    if res.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/ie7615-generative-project")
    if (REPO_DIR / ".git").exists():
        print(_git("-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH))
    else:
        _git("clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("repo:", REPO_DIR, "| commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
from src.utils import load_config, get_paths, set_seed
cfg = load_config()
paths = get_paths(cfg)
set_seed(cfg["seed"])
print("artifact root:", paths["root"])

In [ ]:
import pandas as pd
from src.bert_adapter import train_bert_adapter
from src.utils import load_json

RETRAIN = False
metrics_path = paths["milestone2_logs"] / "bert_adapter_metrics.json"
log_path = paths["milestone2_logs"] / "bert_training_log.csv"
assert paths["manifest"].exists(), "Run notebook 01 first (manifest missing on Drive)."

if RETRAIN or not (paths["bert_adapter_ckpt"].exists() and metrics_path.exists() and log_path.exists()):
    _, history, m = train_bert_adapter(cfg, paths)
    df_log = pd.DataFrame(history)
else:
    print("[skip] adapter already trained; set RETRAIN = True to retrain")
    m, df_log = load_json(metrics_path), pd.read_csv(log_path)
print(f"pairs: train {m['n_train_pairs']} ({m['n_train_images']} images) | val {m['n_val_pairs']} "
      f"({m['n_val_images']} images) | excluded images {m['excluded_image_ids']}")
display(df_log)

## Adapter vs. baselines (validation, held-out images)

In [ ]:
best = m["best_epoch"]
table = pd.DataFrame(
    [{"method": k, **v} for k, v in m["baselines_val"].items()]
    + [{"method": f"adapter (epoch {best['epoch']})", "mse_all": best["val_mse_all"], "cos_all": best["val_cos_all"],
        "mse_content": best["val_mse_content"], "cos_content": best["val_cos_content"]}]
).set_index("method").round(4)
table.to_csv(paths["milestone2_logs"] / "bert_adapter_vs_baselines.csv")
display(table)

In [ ]:
import matplotlib.pyplot as plt

base = m["baselines_val"]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(df_log["epoch"], df_log["train_mse_all"], marker="o", label="train")
axes[0].plot(df_log["epoch"], df_log["val_mse_all"], marker="s", label="val")
axes[0].axhline(base["linear_per_position"]["mse_all"], ls="--", c="grey", label="linear per-position baseline (val)")
axes[0].set(title="MSE, all tokens", xlabel="epoch")
for ax, key, title in ((axes[1], "cos_all", "Cosine, all 77 tokens"), (axes[2], "cos_content", "Cosine, content tokens only")):
    ax.plot(df_log["epoch"], df_log[f"train_{key}"], marker="o", label="train")
    ax.plot(df_log["epoch"], df_log[f"val_{key}"], marker="s", label="val")
    ax.axhline(base["mean_clip_embedding"][key], ls=":", c="red", label="mean-embedding baseline")
    ax.axhline(base["linear_per_position"][key], ls="--", c="grey", label="linear per-position baseline")
    ax.set(title=title, xlabel="epoch")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(paths["milestone2_logs"] / "bert_training_curve.png", dpi=150)
plt.show()